# avgpool-reduce — worked example 3: Global average pool that keeps singleton spatial dims (B, C, 1, 1)

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `avgpool-reduce`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

`nn.AdaptiveAvgPool2d((1, 1))` collapses all spatial positions to a mean but KEEPS the height and width axes as size 1, giving `(B, C, 1, 1)`. With einops you express the kept singleton axes by writing `1` on the right side of the reduce pattern, which is cleaner than reducing to `(B, C)` and then unsqueezing.

## Worked solution

**Step 1 — contract.** Input `(B, C, H, W)`; output `(B, C, 1, 1)`. Every spatial position of a channel is averaged into one value, but the spatial axes survive as size-1 dims (useful so the result still broadcasts against `(B, C, H, W)` feature maps).

**Step 2 — recall the full-collapse pattern.** Pure global pool is `b c h w -> b c`. Here `h` and `w` vanish, so both are reduced.

**Step 3 — keep singletons.** To retain the spatial axes as size 1, write the right side as `b c 1 1`. einops understands the literal `1` as 'put a length-1 axis here'. The `h` and `w` are still absent as named axes, so they are still reduced — we just re-insert two singletons.

**Step 4 — use 'mean'.** That makes the reduction an average over the entire spatial extent, exactly global average pooling.

**Step 5 — verify.** `nn.AdaptiveAvgPool2d((1, 1))(x)` returns `(B, C, 1, 1)` directly, so we compare shapes and values.

In [ ]:
import torch as t
import torch.nn as nn
import einops
from torch import Tensor

t.manual_seed(0)

def global_avgpool_keepdim(x: Tensor) -> Tensor:
    return einops.reduce(x, 'b c h w -> b c 1 1', 'mean')

x = t.randn(2, 5, 7, 4)
out = global_avgpool_keepdim(x)
ref = nn.AdaptiveAvgPool2d((1, 1))(x)
print('out shape:', tuple(out.shape))
print('matches AdaptiveAvgPool2d:', bool(t.allclose(out, ref, atol=1e-6)))